In [1]:
import torch
import numpy as np
import open3d as o3d
import open3d.ml.torch as ml3d
import open3d.ml as o3dml

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
print(torch.__version__)
print(torch.cuda.is_available())
print(np.__version__)
print(o3d.__version__)

2.2.2+cu121
True
1.26.4
0.19.0


In [ ]:
import numpy as np
import yaml
from pathlib import Path


class ScannetPrimitivesDet(o3dml.datasets.Scannet):

    def __init__(
        self,
        dataset_path: str,
        scannet_yaml_path: str,
        name: str = "ScannetPrimitivesDet",
        cache_dir: str = "./logs/cache",
        use_cache: bool = False,
        **kwargs,
    ):
        super().__init__(
            dataset_path=dataset_path,
            name=name,
            cache_dir=cache_dir,
            use_cache=use_cache,
            **kwargs,
        )

        self._scannet_yaml_path = scannet_yaml_path
        yaml_path = Path(scannet_yaml_path)
        if not yaml_path.exists():
            raise FileNotFoundError(f"Config file not found: {yaml_path}")

        with open(yaml_path, "r") as f:
            cfg = yaml.safe_load(f)

        self.classes = list(cfg["classes"])
        self.num_classes = len(self.classes)

        semantic_ids = cfg.get(
            "semantic_ids",
            list(range(len(self.classes))),
        )
        cat_ids = cfg.get(
            "cat_ids",
            list(range(len(self.classes))),
        )

        if len(semantic_ids) != self.num_classes:
            raise ValueError(
                f"Length of semantic_ids {len(semantic_ids)} "
                f"does not match number of classes {self.num_classes}"
            )

        if len(cat_ids) != self.num_classes:
            raise ValueError(
                f"Length of cat_ids {len(cat_ids)} "
                f"does not match number of classes {self.num_classes}"
            )

        self.semantic_ids = list(semantic_ids)
        self.cat_ids = np.array(cat_ids, dtype=np.int32)
        self.cat2label = {cat: i for i, cat in enumerate(self.classes)}

        ignored_label = cfg.get("ignored_label", -1)
        self.cat2label["ignored"] = ignored_label

        self.label2cat = {v: k for k, v in self.cat2label.items()}
        self.cat_ids2class = {
            raw_id: i for i, raw_id in enumerate(list(self.cat_ids))
        }

        self.label_to_names = self.get_label_to_names()

    def get_label_to_names(self):
        return self.label2cat

In [5]:
ds = ScannetPrimitivesDet(
    dataset_path="data/scannet_primitives",
    scannet_yaml_path="configs/scannet_primitives.yaml"
)

In [7]:
split = ds.get_split("train")
sample = split.get_data(0)
points = sample["point"].astype(np.float32)
boxes = sample['bounding_boxes']
label = sample["label"].astype(np.int32)

print(f"Points shape: {points.shape}")
print(f"Labels shape: {label.shape}")

Points shape: (12288, 3)
Labels shape: (12288,)


In [9]:
names = ds.label_to_names
num_classes = int(max(names.keys())) + 1
rng = np.random.default_rng(0)
palette = rng.random((num_classes, 3))

In [ ]:
from open3d._ml3d.vis.boundingbox import BoundingBox3D

pcd = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(points))
bbxs = [obj for obj in boxes]
ls = BoundingBox3D.create_lines(bbxs)
o3d.visualization.draw_geometries([pcd, ls])

In [11]:
cfg_file = 'configs/pointpillars_primitives.yml'
cfg = o3dml.utils.Config.load_from_file(cfg_file)
model = ml3d.models.PointPillars(**cfg.model)

In [12]:
CKPT = '/home/kitt/Projects/side_projects/TGU/det_task_20/pointpillars_kitti_202012221652utc.pth'

ckpt = torch.load(CKPT, map_location="cpu")
sd = ckpt.get("model_state_dict", ckpt)
md = model.state_dict()

loadable = {k:v for k,v in sd.items() if k in md and md[k].shape == v.shape}

md.update(loadable)
model.load_state_dict(md, strict=False)

<All keys matched successfully>

In [13]:
pipeline = ml3d.pipelines.ObjectDetection(
    model=model,
    dataset=ds,
    **cfg.pipeline
)

In [14]:
points_xyz = sample["point"].astype(np.float32)   # (N, 3)
data_infer = {
    "point": points_xyz
    }

In [15]:
res = pipeline.run_inference(data_infer)

/home/kitt/Projects/side_projects/TGU/det_task_20/.o3dml/lib/python3.10/site-packages/torch/functional.py:507: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3549.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [16]:
pcd = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(points_xyz))

bbxs = res[0][0:5]
ls = BoundingBox3D.create_lines(bbxs)
o3d.visualization.draw_geometries([pcd, ls], window_name="Pred")

In [17]:
# 1 epoch ~ 10 sec
pipeline.run_train()

training -  loss_cls: 0.484 loss_bbox: 1.781 loss_dir: 0.000 > loss: 2.265: 100%|██████████| 95/95 [00:08<00:00, 10.67it/s] 
validation: 100%|██████████| 142/142 [00:02<00:00, 55.89it/s]
training -  loss_cls: 0.277 loss_bbox: 0.955 loss_dir: 0.000 > loss: 1.232: 100%|██████████| 95/95 [00:08<00:00, 10.93it/s]
validation: 100%|██████████| 142/142 [00:02<00:00, 57.80it/s]
training -  loss_cls: 0.194 loss_bbox: 0.675 loss_dir: 0.000 > loss: 0.869: 100%|██████████| 95/95 [00:08<00:00, 10.95it/s]
validation: 100%|██████████| 142/142 [00:02<00:00, 57.01it/s]
training -  loss_cls: 0.176 loss_bbox: 0.728 loss_dir: 0.000 > loss: 0.904: 100%|██████████| 95/95 [00:08<00:00, 10.77it/s]
validation: 100%|██████████| 142/142 [00:02<00:00, 54.58it/s]
training -  loss_cls: 0.158 loss_bbox: 0.742 loss_dir: 0.000 > loss: 0.900: 100%|██████████| 95/95 [00:09<00:00, 10.56it/s]
validation: 100%|██████████| 142/142 [00:02<00:00, 58.74it/s]
training -  loss_cls: 0.126 loss_bbox: 0.583 loss_dir: 0.000 > loss: 

In [ ]:
# pipeline.run_test()